In [5]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pyomo.environ as pyo

DATA_DIR = Path("stock_data")
TRADING_DAYS_PER_YEAR = 252


def load_prices() -> pd.DataFrame:
    """Long-format aligned adjusted close: one row per (Date, ticker)."""
    wide = pd.read_csv(DATA_DIR / "aligned_adj_close.csv", index_col="Date", parse_dates=True)
    long = wide.melt(ignore_index=False, var_name="ticker", value_name="adj_close").reset_index()
    return long


def load_metadata() -> pd.DataFrame:
    return pd.read_csv(DATA_DIR / "metadata.csv")


def load_prices_with_metadata() -> pd.DataFrame:
    prices = load_prices()
    metadata = load_metadata()
    return prices.merge(metadata, on="ticker", how="left")


def compute_annualized_stats() -> tuple[pd.Series, pd.DataFrame]:
    """Annualized mean returns and covariance matrix, indexed by ticker."""
    wide = pd.read_csv(DATA_DIR / "aligned_adj_close.csv", index_col="Date", parse_dates=True)
    daily_returns = wide.pct_change().dropna()
    mean_returns = daily_returns.mean() * TRADING_DAYS_PER_YEAR
    cov_matrix = daily_returns.cov() * TRADING_DAYS_PER_YEAR
    return mean_returns, cov_matrix


def markowitz_min_variance(target_return: float) -> pd.Series:
    """Long-only, fully-invested minimum-variance portfolio for a given target return."""
    mean_returns, cov_matrix = compute_annualized_stats()
    tickers = list(mean_returns.index)

    model = pyo.ConcreteModel()
    model.tickers = pyo.Set(initialize=tickers)
    model.weight = pyo.Var(model.tickers, bounds=(0, 1))

    model.budget = pyo.Constraint(expr=sum(model.weight[t] for t in tickers) == 1)
    model.target = pyo.Constraint(
        expr=sum(model.weight[t] * mean_returns[t] for t in tickers) >= target_return
    )

    model.variance = pyo.Objective(
        expr=sum(
            model.weight[i] * cov_matrix.loc[i, j] * model.weight[j]
            for i in tickers
            for j in tickers
        ),
        sense=pyo.minimize,
    )

    solver = pyo.SolverFactory("highs")
    result = solver.solve(model)
    pyo.assert_optimal_termination(result)

    weights = pd.Series({t: pyo.value(model.weight[t]) for t in tickers}, name="weight")
    return weights[weights > 1e-6].sort_values(ascending=False)


def plot_allocation_by_stock(weights: pd.Series) -> go.Figure:
    weights = weights.sort_values(ascending=False)
    metadata = load_metadata().set_index("ticker")
    df = weights.rename("weight").rename_axis("ticker").reset_index()
    df = df.merge(metadata[["shortName", "sector"]], on="ticker", how="left")

    fig = px.bar(
        df,
        x="ticker",
        y="weight",
        color="weight",
        color_continuous_scale="Tealgrn",
        text="weight",
        hover_data={"ticker": False, "shortName": True, "sector": True, "weight": ":.1%"},
        title="Portfolio Allocation by Stock",
        template="plotly_white",
    )
    fig.update_traces(texttemplate="%{text:.1%}", textposition="outside")
    fig.update_layout(
        yaxis_tickformat=".0%",
        xaxis_title=None,
        yaxis_title="Weight",
        coloraxis_showscale=False,
        margin=dict(t=60),
    )
    fig.show()
    return fig


def plot_allocation_by_sector(weights: pd.Series) -> go.Figure:
    metadata = load_metadata().set_index("ticker")
    by_sector = weights.groupby(metadata.loc[weights.index, "sector"]).sum().sort_values(ascending=False)
    df = by_sector.rename("weight").rename_axis("sector").reset_index()

    fig = px.pie(
        df,
        names="sector",
        values="weight",
        title="Portfolio Allocation by Sector",
        hole=0.45,
        color_discrete_sequence=px.colors.qualitative.Prism,
        template="plotly_white",
    )
    fig.update_traces(textinfo="label+percent", pull=[0.03] * len(df), hoverinfo="skip", hovertemplate=None)
    fig.update_layout(showlegend=False, margin=dict(t=60))
    fig.show()
    return fig

In [6]:
prices = load_prices_with_metadata()
weights = markowitz_min_variance(target_return=0.20)
plot_allocation_by_stock(weights);
plot_allocation_by_sector(weights);